# Introduction

This notebook implements the **pre-processing stage** of the fetal vein segmentation study. Its scientific role is to construct a controlled family of training-ready image datasets derived from the same original ultrasound acquisitions, so that downstream learning is exposed to defined differences in image conditioning rather than uncontrolled acquisition variability.

The stage addresses the experimental question of whether—and how—classical pre-segmentation operations (point transforms and spatial filters) alter data in ways that materially affect deep model behaviour. Each pipeline applies a transparent, reproducible sequence: optional global intensity adjustments, followed by exactly one spatial filter, while preserving image geometry and patient identifiers required for label pairing.

**Inputs:** raw images in `02_dataset/images/`. **Outputs:** five parallel folders `images_pp_1/` … `images_pp_5/`, plus the unchanged original set used later as a baseline. These artefacts enable the segmentation notebook to train and infer under equivalent protocols across Original and PP1–PP5, and therefore support a fair comparison of preprocessing strategies in the final evaluation stage.


# Dependencies

All third-party and standard-library imports for this notebook are declared here.
Downstream cells must not repeat these imports.

## Import groups

Standard library, numerical computing, image I/O, and optional notebook widgets.


In [8]:
%matplotlib inline

# --- Standard library ---
import io
from pathlib import Path

# --- Numerical computing ---
import numpy as np

# --- Image I/O and visualisation ---
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from PIL import Image

# --- Notebook UI (optional folder selector) ---
import ipywidgets as widgets
from IPython.display import display


The following sections define **reusable functions** for all pipelines (PP1–PP5).
Each PP section contains its own **Configuration** and **Execution** blocks.


# Common Utilities

Shared I/O and representation helpers used by all preprocessing pipelines.
Functions ensure grayscale `uint8` images and stable path handling.



## Generic Utilities


**Provenance:** consolidated I/O helpers (formerly `00_common/00_generic.ipynb`).

**Purpose:** I/O, grayscale conversion, project root resolution, and display helpers.
**Inputs:** File paths, image arrays.
**Outputs:** Prepared `uint8` grayscale arrays; visualisation.
**Dependencies:** NumPy, Matplotlib, ipywidgets.
**Role:** Foundation for all preprocessing pipelines.


## 1.1 Generic Utilities


### `find_project_root()`

Resolves the repository root by locating `02_dataset/` in the parent chain.
Ensures batch paths work in Colab and local Jupyter without hard-coded absolute paths.



In [9]:
def find_project_root(start_path: Path) -> Path:
    """Walk up the directory tree until 02_dataset/ is found (Google Colab workflow)."""
    for candidate_path in [start_path, *start_path.parents]:
        if (candidate_path / "02_dataset").is_dir():
            return candidate_path
    raise FileNotFoundError(
        f"Directory 02_dataset/ not found from {start_path}."
    )


### 1. Function `convert_to_grayscale()`

Converts RGB/RGBA images to grayscale when required.

- preserves spatial dimensions (H, W);
- utiliza a course weighted luminance formula;
- if the image is already 2D, return unchanged.

In [10]:
# ==========================================================
# 1. GRAYSCALE CONVERSION
# ==========================================================

def convert_to_grayscale(input_image):
    """
    Convert RGB/RGBA to grayscale preserving (H, W).
    """

    # If already 2D, no conversion needed
    if input_image.ndim == 2:
        return input_image

    # Extrair canais (assumindo ordem RGB)
    red_channel = input_image[:,  :, 0]
    green_channel = input_image[:,  :, 1]
    blue_channel = input_image[:,  :, 2]

    # Luminance formula (valores usados na UC)
    grayscale_image = 0.299 * red_channel + 0.587 * green_channel + 0.114 * blue_channel

    return grayscale_image


### 2. Function `ensure_uint8_manual()`

Ensures the image uses `uint8` with intensities in [0, 255].

- explicit steps: scale, round, clip, cast;
- avoids out-of-range saturation.


In [11]:
# ==========================================================
# 2. GARANTIR FORMATO UINT8 (PASSO A PASSO)
# ==========================================================

def ensure_uint8_manual(input_image):
    """
    Convert image to uint8 no intervalo [0, 255].
    """

    # Step 1 — if already uint8, return copy
    if input_image.dtype == np.uint8:
        return input_image.copy()

    # Step 2 — use float to avoid intermediate loss
    image_float = input_image.astype(np.float64)

    # Step 3 — if values are in [0, 1], scale to [0, 255]
    maximum_intensity = image_float.max()
    if maximum_intensity <= 1.0:
        scaled_image = image_float * 255.0
    else:
        scaled_image = image_float.copy()

    # Step 4 — round to nearest integer
    rounded_image = np.round(scaled_image)

    # Step 5 — clip to [0, 255]
    clipped_image = np.clip(rounded_image, 0, 255)

    # Step 6 — convert to uint8
    input_image = clipped_image.astype(np.uint8)

    return input_image


### 3. Function `prepare_grayscale_uint8_image()`

Combines preparation steps mais frequentes: grayscale + `uint8`.

- helper reused pelos outros notebooks;
- does not alter content unnecessarily (normalises representation only).

In [14]:
# ==========================================================
# 3. PREPARE IMAGE (GRAYSCALE + UINT8)
# ==========================================================

def prepare_grayscale_uint8_image(input_image):
    """
    Prepare image for classical processing: grayscale e uint8.
    """

    grayscale_image = convert_to_grayscale(input_image)
    input_image = ensure_uint8_manual(grayscale_image)

    return input_image



### 4. Function `load_image()`

Loads an image file from disk e devolve array preparado (`uint8`, grayscale).

- utiliza `matplotlib.image` (como nos worksheets);
- file_path pode ser `str` ou `Path`.

In [15]:
# ==========================================================
# 4. LOAD IMAGE FROM PATH
# ==========================================================

def load_image(file_path):
    """
    Read image from disk e devolve uint8 em grayscale.
    """

    file_path = Path(file_path)

    if not file_path.is_file():
        raise FileNotFoundError(f"File not found: {file_path}")

    # Read with matplotlib (formato do worksheet)
    loaded_image = mpimg.imread(file_path)

    # Prepare for processing
    prepared_image = prepare_grayscale_uint8_image(loaded_image)

    return prepared_image



### 5. Function `display_images_side_by_side()`

Side-by-side display of up to 4 images (visual support during coursework).

- colormap `gray` automatic for 2D images;
- optional titles per image.

In [16]:
# ==========================================================
# 5. SIDE-BY-SIDE IMAGE DISPLAY (UP TO 4)
# ==========================================================

def display_images_side_by_side(images_list, titles=None, show_axes=False, cmap="gray"):
    """
    Mostra entre 1 e 4 images_list na horizontal.
    """

    if not isinstance(images_list, list):
        raise ValueError("O parâmetro 'images_list' deve ser uma lista.")

    image_count = len(images_list)
    if image_count < 1 or image_count > 4:
        raise ValueError("Function accepts between 1 and 4 images.")

    plt.figure(figsize=(5 * image_count, 5))

    for indice in range(image_count):
        plt.subplot(1, image_count, indice + 1)
        current_image = images_list[indice]

        if current_image.ndim == 2:
            plt.imshow(current_image, cmap=cmap)
        else:
            plt.imshow(current_image)

        if titles is not None and indice < len(titles):
            plt.title(titles[indice])

        if not show_axes:
            plt.axis("off")

    plt.tight_layout()
    plt.show()


### 6. Function `print_image_metadata()`

Prints auxiliary array information (shape, dtype, min/max).

- useful to validate loading before filters or transforms.

In [17]:
# ==========================================================
# 6. IMAGE METADATA (AUXILIARY)
# ==========================================================

def print_image_metadata(input_image, label="image"):
    """
    Print dimensions, dtype and intensity range.
    """

    print(f"--- Metadados: {label} ---")
    print(f"Shape: {input_image.shape}")
    print(f"Dtype: {input_image.dtype}")

    if input_image.size == 0:
        print("Empty array.")
        return

    minimum_intensity = input_image.min()
    maximum_intensity = input_image.max()
    print(f"Minimum intensity: {minimum_intensity}")
    print(f"Maximum intensity: {maximum_intensity}")



### 7. Function `create_folder_selector_widget()`

Simple text + button interface to pick a folder in Jupyter.

- optional local dataset exploration;
- returns selected path after confirmation.


In [18]:
# ==========================================================
# 7. SELECIONAR PASTA (INTERFACE NO NOTEBOOK)
# ==========================================================

def create_folder_selector_widget(description="Folder path"):
    """
    Creates widgets for the user to pick a folder.
    Returns Path after clicking Confirm.
    """

    path_field = widgets.Text(
        description=description,
        placeholder=str(Path.cwd()),
        style={"description_width": "initial"},
    )
    confirm_button = widgets.Button(description="Confirm folder")
    area_convolution_result = widgets.Output()

    selected_path = {"valor": None}

    def on_confirm_click(_):
        with area_convolution_result:
            area_convolution_result.clear_output()
            file_path = Path(path_field.value).expanduser().resolve()
            selected_path["valor"] = file_path
            print(f"Selected folder: {file_path}")

    confirm_button.on_click(on_confirm_click)

    display(path_field, confirm_button, area_convolution_result)

    return selected_path



# Global Operations

Optional point-wise transforms applied before spatial filtering.
Enabled per pipeline via flags in the Configuration section.



## Global Operations


## 1.2 Global Operations


## Image preparation (mesmo notebook)

Before point operations, the image must be **grayscale** e em **`uint8`**.


In [19]:
# ==========================================================
# PREPARATION — GRAYSCALE AND UINT8
# ==========================================================

def convert_to_grayscale(input_image):
    """Convert to grayscale (course luminance formula)."""
    if input_image.ndim == 2:
        return input_image
    r = input_image[:,  :, 0]
    g = input_image[:,  :, 1]
    b = input_image[:,  :, 2]
    return 0.299 * r + 0.587 * g + 0.114 * b


def ensure_uint8_manual(input_image):
    """Convert to uint8 em [0, 255] with explicit steps."""
    if input_image.dtype == np.uint8:
        return input_image.copy()
    img_float = input_image.astype(np.float64)
    if img_float.max() <= 1.0:
        img_escalada = img_float * 255.0
    else:
        img_escalada = img_float.copy()
    img_arredondada = np.round(img_escalada)
    img_limitada = np.clip(img_arredondada, 0, 255)
    return img_limitada.astype(np.uint8)


def prepare_grayscale_uint8_image(input_image):
    """Prepare image: grayscale + uint8."""
    return ensure_uint8_manual(convert_to_grayscale(input_image))



## 1. Brightness and contrast

Linear point transformation:

$$g(i,j) = \alpha \cdot f(i,j) + b$$

- $\alpha$ controla o **contraste**;
- $b$ controla o **brilho**;
- values are clipped to [0, 255].

In [20]:
# ==========================================================
# 1. BRIGHTNESS AND CONTRAST (LINEAR TRANSFORMATION)
# ==========================================================

def apply_brightness_contrast(input_image, alpha=1.0, b=0):
    """
    Apply g = alpha * f + b to all pixels (full image).
    """

    input_image = prepare_grayscale_uint8_image(input_image)
    image_height, image_width = input_image.shape
    output_image_array = input_image.copy()

    for row_index in range(image_height):
        for column_index in range(image_width):
            f_ij = float(input_image[row_index, column_index])
            g_ij = alpha * f_ij + b

            if g_ij > 255:
                clipped_value = 255
            elif g_ij < 0:
                clipped_value = 0
            else:
                clipped_value = int(round(g_ij))

            output_image_array[row_index, column_index] = clipped_value

    return output_image_array.astype(np.uint8)



## 2. Image inversion

Each intensity is transformed as $g = 255 - f$.

In [21]:
# ==========================================================
# 2. INTENSITY INVERSION
# ==========================================================

def invert_image(input_image):
    """
    Invert intensities: g = 255 - f (per pixel).
    """

    input_image = prepare_grayscale_uint8_image(input_image)
    image_height, image_width = input_image.shape
    output_image_array = input_image.copy()

    for row_index in range(image_height):
        for column_index in range(image_width):
            f_ij = int(input_image[row_index, column_index])
            output_image_array[row_index, column_index] = 255 - f_ij

    return output_image_array.astype(np.uint8)



## 3. Gamma transformation

Gamma correction (explicit normalisation to [0, 1] then [0, 255]):

$$s = 255 \cdot \left(\frac{r}{255}\right)^{\gamma}$$

In [22]:
# ==========================================================
# 3. GAMMA TRANSFORMATION
# ==========================================================

def apply_gamma_transform(input_image, gamma=1.0):
    """
    Apply gamma correction with explicit per-pixel calculation.
    """

    if gamma <= 0:
        raise ValueError("gamma must be positive.")

    input_image = prepare_grayscale_uint8_image(input_image)
    image_height, image_width = input_image.shape
    output_image_array = np.zeros_like(input_image)

    for row_index in range(image_height):
        for column_index in range(image_width):
            r = float(input_image[row_index, column_index])
            r_normalizado = r / 255.0
            s_normalizado = r_normalizado ** gamma
            s = int(round(s_normalizado * 255.0))
            output_image_array[row_index, column_index] = s

    return output_image_array.astype(np.uint8)



## 4. Logarithmic transformation

$$s = c \cdot \log(1 + r)$$

The convolution_result is linearly rescaled to [0, 255] using the maximum value.

In [23]:
# ==========================================================
# 4. LOGARITHMIC TRANSFORMATION
# ==========================================================

def apply_logarithmic_transform(input_image, c=1.0):
    """
    Apply s = c * log(1 + r) and rescale to uint8.
    """

    input_image = prepare_grayscale_uint8_image(input_image)
    image_height, image_width = input_image.shape

    # Step 1 — compute transformed values in float
    float_map = np.zeros((image_height, image_width), dtype=np.float64)
    for row_index in range(image_height):
        for column_index in range(image_width):
            r = float(input_image[row_index, column_index])
            float_map[row_index, column_index] = c * np.log1p(r)

    # Step 2 — find maximum for normalization
    maximum_value = float(float_map.max())
    if maximum_value <= 0:
        return input_image.copy()

    # Step 3 — convert to uint8
    output_image_array = np.zeros_like(input_image)
    for row_index in range(image_height):
        for column_index in range(image_width):
            valor_normalizado = float_map[row_index, column_index] / maximum_value
            output_image_array[row_index, column_index] = int(round(valor_normalizado * 255.0))

    return output_image_array.astype(np.uint8)



## 5. Exponential transformation

$$s = c \cdot \left(e^{r/255} - 1\right)$$

Explicit rescaling to [0, 255] after computing all values.

In [24]:
# ==========================================================
# 5. EXPONENTIAL TRANSFORMATION
# ==========================================================

def apply_exponential_transform(input_image, c=1.0):
    """
    Apply exponential transformation with final normalisation.
    """

    input_image = prepare_grayscale_uint8_image(input_image)
    image_height, image_width = input_image.shape

    float_map = np.zeros((image_height, image_width), dtype=np.float64)
    for row_index in range(image_height):
        for column_index in range(image_width):
            r = float(input_image[row_index, column_index])
            r_normalizado = r / 255.0
            float_map[row_index, column_index] = c * (np.exp(r_normalizado) - 1.0)

    maximum_value = float(float_map.max())
    if maximum_value <= 0:
        return input_image.copy()

    output_image_array = np.zeros_like(input_image)
    for row_index in range(image_height):
        for column_index in range(image_width):
            valor_normalizado = float_map[row_index, column_index] / maximum_value
            output_image_array[row_index, column_index] = int(round(valor_normalizado * 255.0))

    return output_image_array.astype(np.uint8)



## 6. Histogram equalization

1. Calcular histogram_values $h(g)$ (contagem manual);
2. Calcular histogram_values cumulative_values $H(g)$;
3. Construir lookup_table $T[g] = \mathrm{round}\left(\frac{G}{S} \cdot H[g]\right)$ com $G=256$;
4. Aplicar $g'(i,j) = T[f(i,j)]$ pixel a pixel.


In [25]:
# ==========================================================
# 6. HISTOGRAM AND EQUALIZATION
# ==========================================================

def compute_histogram_manual(input_image):
    """
    Histogram h(g) with explicit bin counts (256 levels).
    """
    input_image = prepare_grayscale_uint8_image(input_image)
    histogram_values = np.zeros(256, dtype=np.int64)
    image_height, image_width = input_image.shape

    for row_index in range(image_height):
        for column_index in range(image_width):
            pixel_intensity = int(input_image[row_index, column_index])
            histogram_values[pixel_intensity] += 1

    return histogram_values


def compute_cumulative_histogram(histogram_values):
    """
    Cumulative histogram H(g).
    """
    cumulative_values = np.zeros(256, dtype=np.int64)
    running_sum = 0
    for g in range(256):
        running_sum += int(histogram_values[g])
        cumulative_values[g] = running_sum
    return cumulative_values


def build_equalization_lookup_table(cumulative_histogram, total_pixel_count, gray_levels=256):
    """
    T[g] = round((G / S) * H[g]).
    """
    lookup_table = np.zeros(gray_levels, dtype=np.uint8)
    for g in range(gray_levels):
        mapped_value = round((gray_levels / total_pixel_count) * cumulative_histogram[g])
        if mapped_value > 255:
            mapped_value = 255
        lookup_table[g] = mapped_value
    return lookup_table


def equalize_histogram(input_image):
    """
    Global histogram equalization (full image).
    """

    input_image = prepare_grayscale_uint8_image(input_image)
    image_height, image_width = input_image.shape
    total_pixel_count = image_height * image_width

    histogram_values = compute_histogram_manual(input_image)
    cumulative_histogram = compute_cumulative_histogram(histogram_values)
    equalization_lookup = build_equalization_lookup_table(cumulative_histogram, total_pixel_count)

    output_image_array = input_image.copy()
    for row_index in range(image_height):
        for column_index in range(image_width):
            original_intensity = int(input_image[row_index, column_index])
            mapped_intensity = equalization_lookup[original_intensity]
            output_image_array[row_index, column_index] = mapped_intensity

    return output_image_array.astype(np.uint8)



# Filter Implementations

Spatial filters implemented with explicit convolution (coursework requirement).
Each PP pipeline selects exactly one filter from this library.



## Filtering


## 1.3 Filtering


## Manual 2D convolution

Shared base for filters linearly separable (mean, Gaussian, Sobel, Laplacian).

- input uses **zero padding**;
- intermediate output em `float` antes de voltar a `uint8`.

In [26]:
# ==========================================================
# MANUAL CONVOLUTION (GRAYSCALE)
# ==========================================================

def apply_manual_convolution(input_image, kernel):
    """
    Convolve 2D uint8 image with 2D float kernel.
    """

    # Validate input (notebook contract)
    if input_image.ndim != 2:
        raise ValueError("Image must be 2D (grayscale).")
    if input_image.dtype != np.uint8:
        raise ValueError("Image must be uint8.")

    kernel = np.asarray(kernel, dtype=np.float64)
    kernel_height, kernel_width = kernel.shape
    pad_y = kernel_height // 2
    pad_x = kernel_width // 2

    # Zero padding
    image_float = input_image.astype(np.float64)
    padded_image = np.pad(image_float, ((pad_y, pad_y), (pad_x, pad_x)), mode="constant")

    image_height, image_width = input_image.shape
    convolution_result = np.zeros((image_height, image_width), dtype=np.float64)

    for row_index in range(image_height):
        for column_index in range(image_width):
            weighted_sum = 0.0
            for kernel_row in range(kernel_height):
                for kernel_col in range(kernel_width):
                    pixel_value = padded_image[row_index + kernel_row, column_index + kernel_col]
                    kernel_weight = kernel[kernel_row, kernel_col]
                    weighted_sum += pixel_value * kernel_weight
            convolution_result[row_index, column_index] = weighted_sum

    return convolution_result


def clip_float_to_uint8(float_map):
    """Clip float values to uint8 range [0, 255]."""
    rounded_map = np.round(float_map)
    clipped_map = np.clip(rounded_map, 0, 255)
    return clipped_map.astype(np.uint8)


### Low-pass — Average (mean)

Kernel de mean $1/(N \times N)$ com odd size (ex.: 3×3).

In [27]:
# ==========================================================
# FILTRO AVERAGE (PASSA-BAIXO)
# ==========================================================

def apply_average_filter(input_image):
    """
    Mean filter — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    # Janela 3x3 (fixed size per notebook contract)
    window_size = 3

    # Construir kernel de mean explicitamente
    kernel = np.ones((window_size, window_size), dtype=np.float64)
    weight_sum = window_size * window_size
    kernel = kernel / weight_sum

    filtered_map = apply_manual_convolution(input_image, kernel)
    return clip_float_to_uint8(filtered_map)


### Low-pass — Median (median)

Para cada janela, ordena valores e escolhe o elemento central.

In [28]:
# ==========================================================
# FILTRO MEDIAN (PASSA-BAIXO)
# ==========================================================

def apply_median_filter(input_image):
    """
    Median filter — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    window_size = 3
    pad = window_size // 2
    image_height, image_width = input_image.shape
    padded_image = np.pad(input_image, pad, mode="edge")
    output_image_array = np.zeros_like(input_image)

    for row_index in range(image_height):
        for column_index in range(image_width):
            window_pixels = padded_image[
                row_index : row_index + window_size,
                column_index : column_index + window_size,
            ]
            sorted_values = window_pixels.reshape(-1).tolist()
            sorted_values.sort()
            median_index = len(sorted_values) // 2
            output_image_array[row_index, column_index] = sorted_values[median_index]

    return output_image_array.astype(np.uint8)


### Low-pass — Gaussian

Kernel Gaussian 2D built explicitly (sem built-in smoothing functions).

In [29]:
# ==========================================================
# FILTRO GAUSSIAN (PASSA-BAIXO)
# ==========================================================

def apply_gaussian_filter(input_image):
    """
    Filtro Gaussian — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    window_size = 3
    sigma = 1.0
    centro = window_size // 2
    kernel = np.zeros((window_size, window_size), dtype=np.float64)
    kernel_sum = 0.0

    for i in range(window_size):
        for j in range(window_size):
            x = float(j - centro)
            y = float(i - centro)
            kernel_weight = np.exp(-(x * x + y * y) / (2.0 * sigma * sigma))
            kernel[i, j] = kernel_weight
            kernel_sum += kernel_weight

    kernel = kernel / kernel_sum

    filtered_map = apply_manual_convolution(input_image, kernel)
    return clip_float_to_uint8(filtered_map)


### High-pass — Sobel

Gradient in directions $G_x$ e $G_y$; magnitude $|G| = \sqrt{G_x^2 + G_y^2}$.

In [30]:
# ==========================================================
# FILTRO SOBEL (PASSA-ALTO)
# ==========================================================

def apply_sobel_filter(input_image):
    """
    Sobel operator — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    kernel_gx = np.array(
        [
            [-1.0, 0.0, 1.0],
            [-2.0, 0.0, 2.0],
            [-1.0, 0.0, 1.0],
        ],
        dtype=np.float64,
    )

    kernel_gy = np.array(
        [
            [-1.0, -2.0, -1.0],
            [0.0, 0.0, 0.0],
            [1.0, 2.0, 1.0],
        ],
        dtype=np.float64,
    )

    gradient_x = apply_manual_convolution(input_image, kernel_gx)
    gradient_y = apply_manual_convolution(input_image, kernel_gy)

    image_height, image_width = input_image.shape
    magnitude = np.zeros((image_height, image_width), dtype=np.float64)

    for row_index in range(image_height):
        for column_index in range(image_width):
            gx = gradient_x[row_index, column_index]
            gy = gradient_y[row_index, column_index]
            magnitude[row_index, column_index] = np.sqrt(gx * gx + gy * gy)

    return clip_float_to_uint8(magnitude)


### High-pass — Laplacian

Discrete 3×3 kernel; uses absolute filter response and rescales to `uint8`.

In [31]:
# ==========================================================
# FILTRO LAPLACIAN (PASSA-ALTO)
# ==========================================================

def apply_laplacian_filter(input_image):
    """
    Laplacian discreto — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    kernel_laplacian = np.array(
        [
            [0.0, 1.0, 0.0],
            [1.0, -4.0, 1.0],
            [0.0, 1.0, 0.0],
        ],
        dtype=np.float64,
    )

    filter_response = apply_manual_convolution(input_image, kernel_laplacian)

    image_height, image_width = input_image.shape
    absolute_response = np.zeros((image_height, image_width), dtype=np.float64)

    for row_index in range(image_height):
        for column_index in range(image_width):
            absolute_response[row_index, column_index] = abs(filter_response[row_index, column_index])

    maximum_value = float(absolute_response.max())
    if maximum_value <= 0:
        return input_image.copy()

    scaled_response = (absolute_response / maximum_value) * 255.0
    return clip_float_to_uint8(scaled_response)


# PP1 — Average Filter

Preprocessing pipeline **PP1**: optional global operations, **Average** filter, output folder `02_dataset/images_pp_1/`.


## Configuration

Adjust global-operation flags and `PIPELINE_NUMBER` below.


In [32]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION (PP1)
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

PIPELINE_NUMBER = 1
PIPELINE_FILTER_NAME = "Average"



### Paths

**Inputs:** `INPUT_DIR` under project root.
**Outputs:** `OUTPUT_DIR` = `images_pp_{num}/`.


In [33]:
# ==========================================================
# PATHS (PP1)
# ==========================================================

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = PROJECT_ROOT / "02_dataset" / f"images_pp_{PIPELINE_NUMBER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")



Project root: C:\AA_trabalho\fetal_vein_segmentation
Input: C:\AA_trabalho\fetal_vein_segmentation\02_dataset\images
Output:  C:\AA_trabalho\fetal_vein_segmentation\02_dataset\images_pp_1


## Execution

1. Load image (`load_image`).
2. Apply configured global operations.
3. Apply **Average** filter.
4. Validate geometry; save with `_PP_PL_N` suffix.


In [34]:
# ==========================================================
# EXECUTION — PP1 (Average)
# ==========================================================

def apply_configured_global_operations(input_image):
    """Apply only enabled point operations for this pipeline."""
    processed_image = input_image
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        processed_image = apply_brightness_contrast(
            processed_image, alpha=alpha, b=beta
        )
    if APPLY_GAMMA:
        processed_image = apply_gamma_transform(processed_image, gamma=GAMMA_VALUE)
    return processed_image

def apply_pipeline_filter(input_image):
    return apply_average_filter(input_image)

input_files = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
)

if len(input_files) == 0:
    print(f"WARNING: no images found in {INPUT_DIR}")
else:
    print(f"Processing {len(input_files)} image(s) for PP1...")

for input_path in input_files:
    original_image = load_image(input_path)
    original_height, original_width = original_image.shape
    globally_processed_image = apply_configured_global_operations(original_image)
    output_image = apply_pipeline_filter(globally_processed_image)
    output_height, output_width = output_image.shape
    if (output_height, output_width) != (original_height, original_width):
        raise ValueError(
            f"Geometry changed for {input_path.name}: "
            f"{original_height}x{original_width} -> {output_height}x{output_width}"
        )
    output_filename = f"{input_path.stem}_PP_PL_{PIPELINE_NUMBER}.png"
    output_path = OUTPUT_DIR / output_filename
    Image.fromarray(output_image, mode="L").save(output_path)
    print(f"Saved: {output_path.name}")

print("Pipeline 01 complete.")



Processing 152 image(s) for PP1...
Saved: P0100_IMG1_PP_PL_1.png
Saved: P0101_IMG1_PP_PL_1.png
Saved: P0102_IMG1_PP_PL_1.png
Saved: P0103_IMG1_PP_PL_1.png
Saved: P0104_IMG1_PP_PL_1.png
Saved: P0105_IMG1_PP_PL_1.png
Saved: P0106_IMG1_PP_PL_1.png
Saved: P0107_IMG1_PP_PL_1.png
Saved: P0108_IMG1_PP_PL_1.png
Saved: P0109_IMG1_PP_PL_1.png
Saved: P010_IMG1_PP_PL_1.png
Saved: P0110_IMG1_PP_PL_1.png
Saved: P0111_IMG1_PP_PL_1.png
Saved: P0112_IMG1_PP_PL_1.png
Saved: P0114_IMG1_PP_PL_1.png
Saved: P0115_IMG1_PP_PL_1.png
Saved: P0116_IMG1_PP_PL_1.png
Saved: P0117_IMG1_PP_PL_1.png
Saved: P0118_IMG1_PP_PL_1.png
Saved: P0119_IMG1_PP_PL_1.png
Saved: P0120_IMG1_PP_PL_1.png
Saved: P0122_IMG1_PP_PL_1.png
Saved: P0123_IMG1_PP_PL_1.png
Saved: P0124_IMG1_PP_PL_1.png
Saved: P0126_IMG1_PP_PL_1.png
Saved: P0127_IMG1_PP_PL_1.png
Saved: P0128_IMG1_PP_PL_1.png
Saved: P0129_IMG1_PP_PL_1.png
Saved: P012_IMG1_PP_PL_1.png
Saved: P0130_IMG1_PP_PL_1.png
Saved: P0131_IMG1_PP_PL_1.png
Saved: P0132_IMG1_PP_PL_1.png
Saved: 

## Conclusions

PP1 complete. Outputs in `02_dataset/images_pp_1/`. Use `DATASET_FOLDER = "images_pp_1"` in segmentation when required.


# PP2 — Median Filter

Preprocessing pipeline **PP2**: optional global operations, **Median** filter, output folder `02_dataset/images_pp_2/`.


## Configuration

Adjust global-operation flags and `PIPELINE_NUMBER` below.


In [35]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION (PP2)
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

PIPELINE_NUMBER = 2
PIPELINE_FILTER_NAME = "Median"



### Paths

**Inputs:** `INPUT_DIR` under project root.
**Outputs:** `OUTPUT_DIR` = `images_pp_{num}/`.


In [36]:
# ==========================================================
# PATHS (PP2)
# ==========================================================

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = PROJECT_ROOT / "02_dataset" / f"images_pp_{PIPELINE_NUMBER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")



Project root: C:\AA_trabalho\fetal_vein_segmentation
Input: C:\AA_trabalho\fetal_vein_segmentation\02_dataset\images
Output:  C:\AA_trabalho\fetal_vein_segmentation\02_dataset\images_pp_2


## Execution

1. Load image (`load_image`).
2. Apply configured global operations.
3. Apply **Median** filter.
4. Validate geometry; save with `_PP_PL_N` suffix.


In [37]:
# ==========================================================
# EXECUTION — PP2 (Median)
# ==========================================================

def apply_configured_global_operations(input_image):
    """Apply only enabled point operations for this pipeline."""
    processed_image = input_image
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        processed_image = apply_brightness_contrast(
            processed_image, alpha=alpha, b=beta
        )
    if APPLY_GAMMA:
        processed_image = apply_gamma_transform(processed_image, gamma=GAMMA_VALUE)
    return processed_image

def apply_pipeline_filter(input_image):
    return apply_median_filter(input_image)

input_files = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
)

if len(input_files) == 0:
    print(f"WARNING: no images found in {INPUT_DIR}")
else:
    print(f"Processing {len(input_files)} image(s) for PP2...")

for input_path in input_files:
    original_image = load_image(input_path)
    original_height, original_width = original_image.shape
    globally_processed_image = apply_configured_global_operations(original_image)
    output_image = apply_pipeline_filter(globally_processed_image)
    output_height, output_width = output_image.shape
    if (output_height, output_width) != (original_height, original_width):
        raise ValueError(
            f"Geometry changed for {input_path.name}: "
            f"{original_height}x{original_width} -> {output_height}x{output_width}"
        )
    output_filename = f"{input_path.stem}_PP_PL_{PIPELINE_NUMBER}.png"
    output_path = OUTPUT_DIR / output_filename
    Image.fromarray(output_image, mode="L").save(output_path)
    print(f"Saved: {output_path.name}")

print("Pipeline 02 complete.")



Processing 152 image(s) for PP2...
Saved: P0100_IMG1_PP_PL_2.png
Saved: P0101_IMG1_PP_PL_2.png
Saved: P0102_IMG1_PP_PL_2.png
Saved: P0103_IMG1_PP_PL_2.png
Saved: P0104_IMG1_PP_PL_2.png
Saved: P0105_IMG1_PP_PL_2.png
Saved: P0106_IMG1_PP_PL_2.png
Saved: P0107_IMG1_PP_PL_2.png
Saved: P0108_IMG1_PP_PL_2.png
Saved: P0109_IMG1_PP_PL_2.png
Saved: P010_IMG1_PP_PL_2.png
Saved: P0110_IMG1_PP_PL_2.png
Saved: P0111_IMG1_PP_PL_2.png
Saved: P0112_IMG1_PP_PL_2.png
Saved: P0114_IMG1_PP_PL_2.png
Saved: P0115_IMG1_PP_PL_2.png
Saved: P0116_IMG1_PP_PL_2.png
Saved: P0117_IMG1_PP_PL_2.png
Saved: P0118_IMG1_PP_PL_2.png
Saved: P0119_IMG1_PP_PL_2.png
Saved: P0120_IMG1_PP_PL_2.png
Saved: P0122_IMG1_PP_PL_2.png
Saved: P0123_IMG1_PP_PL_2.png
Saved: P0124_IMG1_PP_PL_2.png
Saved: P0126_IMG1_PP_PL_2.png
Saved: P0127_IMG1_PP_PL_2.png
Saved: P0128_IMG1_PP_PL_2.png
Saved: P0129_IMG1_PP_PL_2.png
Saved: P012_IMG1_PP_PL_2.png
Saved: P0130_IMG1_PP_PL_2.png
Saved: P0131_IMG1_PP_PL_2.png
Saved: P0132_IMG1_PP_PL_2.png
Saved: 

## Conclusions

PP2 complete. Outputs in `02_dataset/images_pp_2/`. Use `DATASET_FOLDER = "images_pp_2"` in segmentation when required.


# PP3 — Gaussian Filter

Preprocessing pipeline **PP3**: optional global operations, **Gaussian** filter, output folder `02_dataset/images_pp_3/`.


## Configuration

Adjust global-operation flags and `PIPELINE_NUMBER` below.


In [38]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION (PP3)
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

PIPELINE_NUMBER = 3
PIPELINE_FILTER_NAME = "Gaussian"



### Paths

Resolved relative to repository root (`02_dataset/`).


In [39]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = PROJECT_ROOT / "02_dataset" / f"images_pp_{PIPELINE_NUMBER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALID_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")



## Execution

1. Load image (`load_image`).
2. Apply configured global operations.
3. Apply **Gaussian** filter.
4. Validate geometry; save with `_PP_PL_N` suffix.


In [40]:
# ==========================================================
# EXECUTION — PP3 (Gaussian)
# ==========================================================

def apply_configured_global_operations(input_image):
    """Apply only enabled point operations for this pipeline."""
    processed_image = input_image
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        processed_image = apply_brightness_contrast(
            processed_image, alpha=alpha, b=beta
        )
    if APPLY_GAMMA:
        processed_image = apply_gamma_transform(processed_image, gamma=GAMMA_VALUE)
    return processed_image

def apply_pipeline_filter(input_image):
    return apply_gaussian_filter(input_image)

input_files = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
)

if len(input_files) == 0:
    print(f"WARNING: no images found in {INPUT_DIR}")
else:
    print(f"Processing {len(input_files)} image(s) for PP3...")

for input_path in input_files:
    original_image = load_image(input_path)
    original_height, original_width = original_image.shape
    globally_processed_image = apply_configured_global_operations(original_image)
    output_image = apply_pipeline_filter(globally_processed_image)
    output_height, output_width = output_image.shape
    if (output_height, output_width) != (original_height, original_width):
        raise ValueError(
            f"Geometry changed for {input_path.name}: "
            f"{original_height}x{original_width} -> {output_height}x{output_width}"
        )
    output_filename = f"{input_path.stem}_PP_PL_{PIPELINE_NUMBER}.png"
    output_path = OUTPUT_DIR / output_filename
    Image.fromarray(output_image, mode="L").save(output_path)
    print(f"Saved: {output_path.name}")

print("Pipeline 03 complete.")



Processing 152 image(s) for PP3...
Saved: P0100_IMG1_PP_PL_3.png
Saved: P0101_IMG1_PP_PL_3.png
Saved: P0102_IMG1_PP_PL_3.png
Saved: P0103_IMG1_PP_PL_3.png
Saved: P0104_IMG1_PP_PL_3.png
Saved: P0105_IMG1_PP_PL_3.png
Saved: P0106_IMG1_PP_PL_3.png
Saved: P0107_IMG1_PP_PL_3.png
Saved: P0108_IMG1_PP_PL_3.png
Saved: P0109_IMG1_PP_PL_3.png
Saved: P010_IMG1_PP_PL_3.png
Saved: P0110_IMG1_PP_PL_3.png
Saved: P0111_IMG1_PP_PL_3.png
Saved: P0112_IMG1_PP_PL_3.png
Saved: P0114_IMG1_PP_PL_3.png
Saved: P0115_IMG1_PP_PL_3.png
Saved: P0116_IMG1_PP_PL_3.png
Saved: P0117_IMG1_PP_PL_3.png
Saved: P0118_IMG1_PP_PL_3.png
Saved: P0119_IMG1_PP_PL_3.png
Saved: P0120_IMG1_PP_PL_3.png
Saved: P0122_IMG1_PP_PL_3.png
Saved: P0123_IMG1_PP_PL_3.png
Saved: P0124_IMG1_PP_PL_3.png
Saved: P0126_IMG1_PP_PL_3.png
Saved: P0127_IMG1_PP_PL_3.png
Saved: P0128_IMG1_PP_PL_3.png
Saved: P0129_IMG1_PP_PL_3.png
Saved: P012_IMG1_PP_PL_3.png
Saved: P0130_IMG1_PP_PL_3.png
Saved: P0131_IMG1_PP_PL_3.png
Saved: P0132_IMG1_PP_PL_3.png
Saved: 

## Conclusions

PP3 complete. Outputs in `02_dataset/images_pp_3/`. Use `DATASET_FOLDER = "images_pp_3"` in segmentation when required.


# PP4 — Sobel Filter

Preprocessing pipeline **PP4**: optional global operations, **Sobel** filter, output folder `02_dataset/images_pp_4/`.


## Configuration

Adjust global-operation flags and `PIPELINE_NUMBER` below.


In [41]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION (PP4)
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

PIPELINE_NUMBER = 4
PIPELINE_FILTER_NAME = "Sobel"



### Paths

Resolved relative to repository root (`02_dataset/`).


In [42]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = PROJECT_ROOT / "02_dataset" / f"images_pp_{PIPELINE_NUMBER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALID_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")



## Execution

1. Load image (`load_image`).
2. Apply configured global operations.
3. Apply **Sobel** filter.
4. Validate geometry; save with `_PP_PL_N` suffix.


In [43]:
# ==========================================================
# EXECUTION — PP4 (Sobel)
# ==========================================================

def apply_configured_global_operations(input_image):
    """Apply only enabled point operations for this pipeline."""
    processed_image = input_image
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        processed_image = apply_brightness_contrast(
            processed_image, alpha=alpha, b=beta
        )
    if APPLY_GAMMA:
        processed_image = apply_gamma_transform(processed_image, gamma=GAMMA_VALUE)
    return processed_image

def apply_pipeline_filter(input_image):
    return apply_sobel_filter(input_image)

input_files = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
)

if len(input_files) == 0:
    print(f"WARNING: no images found in {INPUT_DIR}")
else:
    print(f"Processing {len(input_files)} image(s) for PP4...")

for input_path in input_files:
    original_image = load_image(input_path)
    original_height, original_width = original_image.shape
    globally_processed_image = apply_configured_global_operations(original_image)
    output_image = apply_pipeline_filter(globally_processed_image)
    output_height, output_width = output_image.shape
    if (output_height, output_width) != (original_height, original_width):
        raise ValueError(
            f"Geometry changed for {input_path.name}: "
            f"{original_height}x{original_width} -> {output_height}x{output_width}"
        )
    output_filename = f"{input_path.stem}_PP_PL_{PIPELINE_NUMBER}.png"
    output_path = OUTPUT_DIR / output_filename
    Image.fromarray(output_image, mode="L").save(output_path)
    print(f"Saved: {output_path.name}")

print("Pipeline 04 complete.")



Processing 152 image(s) for PP4...
Saved: P0100_IMG1_PP_PL_4.png
Saved: P0101_IMG1_PP_PL_4.png
Saved: P0102_IMG1_PP_PL_4.png
Saved: P0103_IMG1_PP_PL_4.png
Saved: P0104_IMG1_PP_PL_4.png
Saved: P0105_IMG1_PP_PL_4.png
Saved: P0106_IMG1_PP_PL_4.png
Saved: P0107_IMG1_PP_PL_4.png
Saved: P0108_IMG1_PP_PL_4.png
Saved: P0109_IMG1_PP_PL_4.png
Saved: P010_IMG1_PP_PL_4.png
Saved: P0110_IMG1_PP_PL_4.png
Saved: P0111_IMG1_PP_PL_4.png
Saved: P0112_IMG1_PP_PL_4.png
Saved: P0114_IMG1_PP_PL_4.png
Saved: P0115_IMG1_PP_PL_4.png
Saved: P0116_IMG1_PP_PL_4.png
Saved: P0117_IMG1_PP_PL_4.png
Saved: P0118_IMG1_PP_PL_4.png
Saved: P0119_IMG1_PP_PL_4.png
Saved: P0120_IMG1_PP_PL_4.png
Saved: P0122_IMG1_PP_PL_4.png
Saved: P0123_IMG1_PP_PL_4.png
Saved: P0124_IMG1_PP_PL_4.png
Saved: P0126_IMG1_PP_PL_4.png
Saved: P0127_IMG1_PP_PL_4.png
Saved: P0128_IMG1_PP_PL_4.png
Saved: P0129_IMG1_PP_PL_4.png
Saved: P012_IMG1_PP_PL_4.png
Saved: P0130_IMG1_PP_PL_4.png
Saved: P0131_IMG1_PP_PL_4.png
Saved: P0132_IMG1_PP_PL_4.png
Saved: 

## Conclusions

PP4 complete. Outputs in `02_dataset/images_pp_4/`. Use `DATASET_FOLDER = "images_pp_4"` in segmentation when required.


# PP5 — Laplacian Filter

Preprocessing pipeline **PP5**: optional global operations, **Laplacian** filter, output folder `02_dataset/images_pp_5/`.


## Configuration

Adjust global-operation flags and `PIPELINE_NUMBER` below.


In [44]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION (PP5)
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

PIPELINE_NUMBER = 5
PIPELINE_FILTER_NAME = "Laplacian"



### Paths

Resolved relative to repository root (`02_dataset/`).


In [45]:
PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = PROJECT_ROOT / "02_dataset" / f"images_pp_{PIPELINE_NUMBER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALID_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")



## Execution

1. Load image (`load_image`).
2. Apply configured global operations.
3. Apply **Laplacian** filter.
4. Validate geometry; save with `_PP_PL_N` suffix.


In [46]:
# ==========================================================
# EXECUTION — PP5 (Laplacian)
# ==========================================================

def apply_configured_global_operations(input_image):
    """Apply only enabled point operations for this pipeline."""
    processed_image = input_image
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        processed_image = apply_brightness_contrast(
            processed_image, alpha=alpha, b=beta
        )
    if APPLY_GAMMA:
        processed_image = apply_gamma_transform(processed_image, gamma=GAMMA_VALUE)
    return processed_image

def apply_pipeline_filter(input_image):
    return apply_laplacian_filter(input_image)

input_files = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
)

if len(input_files) == 0:
    print(f"WARNING: no images found in {INPUT_DIR}")
else:
    print(f"Processing {len(input_files)} image(s) for PP5...")

for input_path in input_files:
    original_image = load_image(input_path)
    original_height, original_width = original_image.shape
    globally_processed_image = apply_configured_global_operations(original_image)
    output_image = apply_pipeline_filter(globally_processed_image)
    output_height, output_width = output_image.shape
    if (output_height, output_width) != (original_height, original_width):
        raise ValueError(
            f"Geometry changed for {input_path.name}: "
            f"{original_height}x{original_width} -> {output_height}x{output_width}"
        )
    output_filename = f"{input_path.stem}_PP_PL_{PIPELINE_NUMBER}.png"
    output_path = OUTPUT_DIR / output_filename
    Image.fromarray(output_image, mode="L").save(output_path)
    print(f"Saved: {output_path.name}")

print("Pipeline 05 complete.")



Processing 152 image(s) for PP5...
Saved: P0100_IMG1_PP_PL_5.png
Saved: P0101_IMG1_PP_PL_5.png
Saved: P0102_IMG1_PP_PL_5.png
Saved: P0103_IMG1_PP_PL_5.png
Saved: P0104_IMG1_PP_PL_5.png
Saved: P0105_IMG1_PP_PL_5.png
Saved: P0106_IMG1_PP_PL_5.png
Saved: P0107_IMG1_PP_PL_5.png
Saved: P0108_IMG1_PP_PL_5.png
Saved: P0109_IMG1_PP_PL_5.png
Saved: P010_IMG1_PP_PL_5.png
Saved: P0110_IMG1_PP_PL_5.png
Saved: P0111_IMG1_PP_PL_5.png
Saved: P0112_IMG1_PP_PL_5.png
Saved: P0114_IMG1_PP_PL_5.png
Saved: P0115_IMG1_PP_PL_5.png
Saved: P0116_IMG1_PP_PL_5.png
Saved: P0117_IMG1_PP_PL_5.png
Saved: P0118_IMG1_PP_PL_5.png
Saved: P0119_IMG1_PP_PL_5.png
Saved: P0120_IMG1_PP_PL_5.png
Saved: P0122_IMG1_PP_PL_5.png
Saved: P0123_IMG1_PP_PL_5.png
Saved: P0124_IMG1_PP_PL_5.png
Saved: P0126_IMG1_PP_PL_5.png
Saved: P0127_IMG1_PP_PL_5.png
Saved: P0128_IMG1_PP_PL_5.png
Saved: P0129_IMG1_PP_PL_5.png
Saved: P012_IMG1_PP_PL_5.png
Saved: P0130_IMG1_PP_PL_5.png
Saved: P0131_IMG1_PP_PL_5.png
Saved: P0132_IMG1_PP_PL_5.png
Saved: 

## Conclusions

PP5 complete. Outputs in `02_dataset/images_pp_5/`. Use `DATASET_FOLDER = "images_pp_5"` in segmentation when required.


# 8. Outputs

Generated folders: `02_dataset/images_pp_1/` … `images_pp_5/`.


# 9. Conclusions

Datasets are consumed by `02_segmentation.ipynb` (`DATASET_FOLDER`).
Evaluation uses predictions from `03_evaluation.ipynb`.
